In [ ]:
import os, sys
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import cooler
import matplotlib
import matplotlib.pylab as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")
from PyComplexHeatmap import *
matplotlib.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['figure.dpi'] = 80
matplotlib.rcParams['savefig.dpi']=300

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/BICAN_pseudobulk"
indir = None
mode='raw'
res=100000
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv"
prefix='Subclass+Region' # used to define indir: {prefix}_compartment
group_col='Subclass' # Region # used to define figdir, and cell_types (group_order), match the col in metadata.columns
parent_col="Class" #used to sort the group_col (MajorType)
sort_cols=None
stratify_col = None #Subclass
stratify_figdir=None
palette_path = "/home/x-wding2/Projects/BG/metadata/HMBA_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
figdir=None
cpu = 16
kernel_name = "m3c"
excluded_groups=['Low quality cells','doublet','Unk']
# stratify
# prefix = "Subclass+Region"
# group_col = "Region"
# parent_col = 'Class'
# stratify_col = "Subclass"

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/compartment"
outname = "1.Class.call_compartment.ipynb"


In [ ]:
# Input:
# (1). cool files: there should be {ct}.mcool (raw) or {ct}.Q.cool (imputed) under {indir}, for example, {ct} can be ASC (if group_col is cell type) or E17 (if group_col is Age).
# (2). cg_count_path:
#     To generate CG-count-100k.tsv:
#         bedtools makewindows -g ~/Ref/mm10/mm10_ucsc.nochrM.sizes -w 100000 > mm10.1Mb.bed
#         bedtools nuc -fi ~/Ref/mm10/mm10_ucsc_with_chrL.fa -pattern CG -bed mm10.1Mb.bed > CG-count-100k.tsv
# (3). chrom_size_path, metadata_path (group_col and parent_col should present in columns).
# Output:<br>
#     figdir: {workdir}/compartment/{mode}_{group_col}
#     figdir/merged_compartment.pdf
#     figdir/filtered_bins.npz
#     figdir/compartment_score_pcs.tsv.gz
#     figdir/pc_loadings.npz
#     figdir/saddle.npz
#     figdir/celltype_saddle.pdf
#     figdir/{chrom}_Q.pdf,figdir/{chrom}_compartment.pdf

In [ ]:
if not workdir is None:
    workdir=os.path.expanduser(workdir)
if not indir is None:
    indir=os.path.expanduser(indir)
else:
    assert not workdir is None
    indir=os.path.join(workdir,f"{prefix}_raw") if mode=='raw' else os.path.join(workdir,f"{prefix}_compartment")
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"compartment",f"{mode}_{group_col}")
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)
cg_count_path=os.path.expanduser(cg_count_path)
print(indir,figdir,group_col)

In [ ]:
cpg = pd.read_csv(os.path.expanduser(cg_count_path), header=0, index_col=3, sep='\t')
cpg['CpG_density'] = cpg['13_user_patt_count'] / (cpg['12_seq_len'] - cpg['10_num_N']) #No. of CG devided by length of sequence
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes[chrom_sizes.index!='chrY']
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
# Filter out low quality cells
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))] #remove Exc-Unk
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
metadata.head()

## Define group_order and colors

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not parent_col is None:
    metadata.sort_values(parent_col,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)
if sort_cols is None:
    sorted_groups=metadata[group_col].unique().tolist()
else:
    sorted_groups=metadata.sort_values(sort_cols)[group_col].unique().tolist()
stratify2group={}
group_order=[]
if stratify_col is None:
    for g in sorted_groups:
        sname=g.replace(' ','_')
        if os.path.exists(os.path.join(indir,f'{sname}.mcool')):
            group_order.append(g)
    group_order=group_order+['merged'] # such as different cell types
else:
    stratify_values=metadata[stratify_col].unique().tolist()
    for s in stratify_values:
        s_name=s.replace(' ','_')
        if stratify_figdir is None:
            for g in sorted_groups:
                g_name=g.replace(' ','_')
                if os.path.exists(os.path.join(indir,f'{s_name}+{g_name}.mcool')):
                    if s not in stratify2group:
                        stratify2group[s]=[]
                    stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                    group_order.append(f"{s}+{g}") # such as Subclass+Region: 'Microglia+CaB', 'Microglia+NAC','OPC+CaB'
        else:
            D=metadata.loc[:,[group_col,stratify_col]].drop_duplicates().groupby(stratify_col)[group_col].apply(lambda x:x.unique().tolist()).to_dict()
            for g in D[s]:
                g_name=g.replace(' ','_')
                if os.path.exists(os.path.join(indir,f'{g_name}.mcool')):
                    if s not in stratify2group:
                        stratify2group[s]=[]
                    stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                    group_order.append(f"{g}") # such as Subclass+Region: 'Microglia+CaB', 'Microglia+NAC','OPC+CaB'
            
print(len(group_order),group_order)
print(stratify2group)

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
D1=D[group_col].Hex.to_dict()
group_colors={k: D1[k] for k in D1 if k in sorted_groups}
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    parent_colors['merged']='black'
    if stratify_col is None:
        group2parent=metadata.loc[:,[group_col,parent_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        D1=D[stratify_col].Hex.to_dict()
        stratify_colors={k: D1[k] for k in D1 if k in stratify_values}
        if stratify_figdir is None:
            metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str)
        else:
            metadata['GroupID']=metadata[group_col]
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    group2parent['merged']='merged'
else:
    parent_colors=None
    group2parent=None
print(parent_colors)
print(group_colors)
print(group2parent)

## Filter bins based on 50% and 99% quantile in merged cells (raw concatcts of all cell types)

For pseudobulk analysis, we merged the cells from each group (major type, subtype, or region) by taking the sum of raw matrices or the average of imputed matrices over cells within the group, and only randomly selected 1500 cells to use if the group contained >1500 cells. We also had a group with merged all cell types used in the compartment analysis and embedding comparison. This group contained 5707 cells in total, generated by randomly selected 200 cells with ≥ 100,000 contacts from each major type. We first used the merged contact maps of the 5707 cells and filtered out the 100-kb bins with abnormal coverage. Specifically, the coverage of bin i on chromosome c (denoted as Rc,i)wasd efined as the sum aof the i -th row of the contact matrix of chromo some c .We only kept the bins with coverage between the 99th percentile of Rc and twice the median of Rc minus the 99th percentile of Rc. Contact matrices were normalized by distance, and Pearson ’ s correlation matrices of the normalized matrices were computed. These merged contact maps were used to fit the PCA models per chromosome.

In [ ]:
# Input: indir
# Output variales: filtered_Q, filtered_bins
if stratify_col is None: # and not os.path.exists(os.path.join(figdir,'filtered_bins.npz')):
    filtered_Q = {} #store the filtered Q (remove some bins with abnormal coverage), keys are chroms, values are array
    filtered_bins = {} # an array of True/False to indicate index of filtered bins.
    ct = 'merged' # raw merged, or imputed and merged
    # cool = cooler.Cooler(f'/gale/netapp/entex/HBA/snm3C/compartment_majortype/{ct}/{ct}.raw.mcool::resolutions/100000')
    ncols=5
    nrows=int(np.ceil(chrom_sizes.shape[0] / 5))
    cool = cooler.Cooler(f'{indir}/{ct}.mcool::resolutions/100000')
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 12),constrained_layout = True)
    for i, chrom in enumerate(chrom_sizes.index):
        Q = cool.matrix(balance=False, sparse=True).fetch(chrom).toarray()
        Q = Q - np.diag(np.diag(Q)) #diag are all zeros, excluding self interaction
        rowsum = Q.sum(axis=0)
        thres = [np.percentile(rowsum[rowsum > 0], 50), np.percentile(rowsum[rowsum > 0], 99)]
        thres.append(thres[0] * 2 - thres[1])
        ax = axes.flatten()[i]
        sns.histplot(rowsum, bins=100, ax=ax)
        for i,t in enumerate(thres):
            color='r' if i==2 else 'chocolate'
            ax.plot([t, t], [0, 0.5], c=color,transform=ax.get_xaxis_transform(),linestyle='--')
        ax.set_title(chrom)
        binfilter = (rowsum > thres[-1])
        filtered_bins[chrom]=binfilter
        Q = Q[binfilter][:, binfilter]
        filtered_Q[chrom]=Q
        print(chrom,thres,Q.shape,binfilter.shape)

    for i in range(chrom_sizes.shape[0], axes.flatten().shape[0]):
        axes.flatten()[i].axis('off')
    fig.tight_layout()
    plt.show()
    # np.save(os.path.join(figdir,f'filtered_bins.npy'), np.asarray(filtered_bins,dtype='object')) # array
    # np.save(os.path.join(figdir,f'pc_loadings.npy'), np.asarray(pc_loading_dict,dtype='object')) # array
    np.savez_compressed(os.path.join(figdir,'filtered_bins.npz'), **filtered_bins) # dict #sparse.save_npz(outname, r)
    print(thres)

In [ ]:
# Plot the filtered Q matrix (deacy?, The white bins are filtered regions)
if stratify_col is None:
    cmap='bwr'
    fig, axes = plt.subplots(nrows, 5, figsize=(12,12),dpi=300)
    for i,chrom in enumerate(chrom_sizes.index):
        ax = axes.flatten()[i]
        n_bins = int(chrom_sizes[chrom] // 100000) + 1
        tmp = np.zeros((n_bins, n_bins))
        tmp[np.ix_(filtered_bins[chrom], filtered_bins[chrom])] = filtered_Q[chrom] #Q matrix after filter
        vmax=np.percentile(filtered_Q[chrom], 95)
        ax.imshow(tmp, cmap=cmap, vmin=-1*vmax, vmax=vmax)
        vmax=round(vmax,2)
        ax.set_title(f"{chrom} ({vmax})", fontsize=10)

    for i in range(chrom_sizes.shape[0], axes.flatten().shape[0]):
        axes.flatten()[i].axis('off')

    # mm2inch = 1 / 25.4
    # legend_width_mm=1 #mm
    # ax=plt.gca()
    # ax_extent=ax.get_window_extent()
    # legend_width_pixel=legend_width_mm * mm2inch * ax.figure.dpi
    # legend_width_ratio=legend_width_pixel/ax_extent.width
    # print(legend_width_ratio,ax.figure.dpi)

    # cax = ax.figure.add_axes(rect=[1,0.4,legend_width_ratio,0.2],xmargin=0, ymargin=0)  # unit is fractions of figure width and height, left,bottom,width,height
    # m = plt.cm.ScalarMappable(norm=matplotlib.colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    # cbar = ax.figure.colorbar(m, cax=cax,label="Interaction Frequency",ticks=[vmin,np.round((vmin+vmax)/2,2), vmax])
    # cax.set_ylim([0, vmax])
    # ax.axis('off')

    plt.tight_layout()
    # plt.savefig(f'{figdir}/merged_decay.pdf', transparent=True)
    plt.show()

## Calculate eigenvector (PCA, compartment score) on merged Q

In [ ]:
# Input: filtered_Q, filtered_bins
# Output: matrix_C, xpca_dict, pc_loading_dict
if stratify_col is None: # and not os.path.exists(os.path.join(figdir,'pc_loadings.npz')):
    # comp = pd.read_hdf(f'{indir}compartment_majortype/comp_raw_merged.hdf', key='data')
    matrix_C = {} #spearman correlation matrix
    xpca_dict = {} # a list of PCA for all chromosomes
    pc_loading_dict = {}
    for k,chrom in enumerate(chrom_sizes.index):
        Q = filtered_Q[chrom].copy() # Filtered Q matrix for each chr
        decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])]) #np.diag(*,k), k: how many bins above the main diagonal, interaction frequency between the ith bin and i-k bin
        # shape of decay = Q.shape[0] = Q.shape[1]
        E = np.zeros(Q.shape)
        row, col = np.diag_indices(E.shape[0]) #index of diag, np.diag_indices(3)= (array([0, 1, 2]), array([0, 1, 2]))
        E[row, col] = 1
        for i in range(1, E.shape[0]):
            E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5) #the same diag as decay[i], normalize the contact by dividing the mean contact having the same distance.
        E = E + E.T #distance normalized contact matrix
        C = np.corrcoef(np.log2(E + 0.001)) #Contact matrices were normalized by distance, and Pearson’s correlation matrices of the normalized matrices were computed
        matrix_C[chrom]=C
        binfilter = filtered_bins[chrom]
    #     tmp = comp.loc[ct, (comp.columns.str.split('_').str[0]==chrom)]
    #     tmp.index = [int(xx.split('_')[1]) for xx in tmp.index]
    #     xpca_dict.append(tmp[np.where(binfilter)[0]].values)
        pca = PCA(n_components=2)
        pc = pca.fit_transform(C) # fit the correlation matrix to PCA
        cpgtmp = cpg.loc[cpg['#1_usercol']==chrom, 'CpG_density'].values[binfilter]
    #     i = 0 if np.abs(pearsonr(cpgtmp, pc[:,0])[0])>np.abs(pearsonr(cpgtmp, pc[:,1])[0]) else 1
        r = []
        for i in range(2):
            labels, groups = pd.qcut(pc[:,i], 50, labels=False, retbins=True) #pd.qcut(x,q=quantile,),return category and bins, divie pc into 50 eaqual size bins
            # labels is the label (belonging to which group) for each bin (labels.shape=pc.shape, min=0,max=49), groups is the returned bins (shape=51)
            sad = np.array([[E[np.ix_(labels==k, labels==j)].sum() for k in range(50)] for j in range(50)]) # the bins whose row belongs to the k bin and col belongs to the j bin
            # for each chrom, divide the whole E matrix into 50*50 bins, sad is the sum of all interaction of the bins belonging to each 50*50 grid
            count = np.array([[(labels==k).sum()*(labels==j).sum() for k in range(50)] for j in range(50)]) #count is the number of bins in E that are aggregated into each 50*50 grids.
            sad = sad / count
            r.append((sad[:10, :10].sum() + sad[-10:, -10:].sum()) / (sad[:10, -10:].sum() + sad[-10:, :10].sum()))
            # saddle_strength: inter-compartment (AA+BB) interaction devided by intra-comartment (AB+BA), BB: upper left corner; AA: lower right; BA: upper right; AB: lower left;
        if r[0]>r[1]: # automatically determine the PC1 or PC2 as compartment score, sometimes PC1 is correlated the chromosome arms.
            i = 0
        else:
            i = 1
        # automatically determine the PC1 or PC2 as compartment score
        if pearsonr(cpgtmp, pc[:,i])[0]>0: #correlation between PC1,PC2 and the CpG density.
            pc = pc[:,i]
            pc_loading_dict[chrom]=pca.components_[i]
        else:
            pc = -pc[:,i]
            pc_loading_dict[chrom]=-pca.components_[i]
        xpca_dict[chrom]=pc
    np.savez_compressed(os.path.join(figdir,'pc_loadings.npz'), **pc_loading_dict)
# END

## Plot Compartments for merged cells (all cell types)

In [ ]:
def plot_compartment(corr_matrix=None,x_pca=None,n_bins=None,ax=None,subplot_spec=None,title=None,vmin=None,vmax=None,cmap='bwr',color='black',xlabel=None,ylabel=None):
    if ax is None:
        ax=plt.gca()
    if subplot_spec is None:
        gs = ax.figure.add_gridspec(2,1,height_ratios=[3,0.6],hspace=0,wspace=0)
    else:
        gs = matplotlib.gridspec.GridSpecFromSubplotSpec(2,1,subplot_spec=subplot_spec,height_ratios=[3,0.6],hspace=0.0,wspace=0)
    ax1 = ax.figure.add_subplot(gs[0,0])
                
    ax1.imshow(corr_matrix, cmap=cmap, vmin=vmin, vmax=vmax,rasterized=True)
               # origin='upper',aspect='auto') #extent=[0,n_bins,0,n_bins],
    ax1.set_xticks([])
    # ax1.set_yticks([])
    if not xlabel is None:
        ax1.set_xlabel(xlabel,labelpad=0.1) # fontsize=12
    if not ylabel is None:
        ax1.set_ylabel(ylabel,labelpad=0.1) # fontsize=12
    ax1.set_xlim(0,n_bins)
    # ax1.set_ylim(0,n_bins)
    # ax1.set_axis_off()
    ax1.set_title(title, fontsize=14,color=color,fontweight='bold')

    ax2 = ax.figure.add_subplot(gs[1,0],sharex=ax1,sharey=None)
    ax2.fill_between(np.arange(n_bins), x_pca, 0, where=x_pca > 0, facecolor='#FF0000', interpolate=True,rasterized=True) #C3
    ax2.fill_between(np.arange(n_bins), x_pca, 0, where=x_pca <= 0, facecolor='#0000FF', interpolate=True,rasterized=True) #C0
    # ax2.set_yticks([])
    ax2.set_ylim([np.percentile(x_pca, 1), np.percentile(x_pca, 99)])
    sns.despine(bottom=True, left=False, top=True,ax=ax2)
    # print(ax1.get_xlim(),ax1.get_ylim(),ax2.get_xlim())

Compartment A represents regions of the genome that tend to interact more frequently with each other (higher contact frequency).

Compartment B represents regions of the genome with lower interaction frequencies, often associated with a more compact chromatin state.

In [ ]:
# Input: matrix_C, xpca_dict, filtered_bins
# from mpl_toolkits.axes_grid1.inset_locator import inset_axes
if stratify_col is None:
    N=chrom_sizes.shape[0]+1
    ncols=min(5,N)
    nrows=int(np.ceil(N / ncols))
    all_contacts=np.hstack([matrix_C[key].flatten() for key in matrix_C])
    v=max(abs(np.percentile(all_contacts,5)),abs(np.percentile(all_contacts,95)))
    vmin,vmax=-1*v,v #use the same vmin and vmax for all chromosomes
    del all_contacts
    cmap='BlueDarkRed12'

    plt.figure(figsize=(3*ncols+0.2,3.6*nrows),dpi=300)
    ax=plt.gca()
    ax.set_axis_off()
    gs = ax.figure.add_gridspec(nrows,ncols,wspace=0.28,hspace=0.1) # width_ratios=self.widths,height_ratios=self.heights,

    row=0
    col=0
    for i,chrom in enumerate(chrom_sizes.index):
        if col >= ncols:
            col=0
            row+=1
        cur_gs=gs[row,col]
        n_bins = int(chrom_sizes[chrom] // 100000) + 1
        corr_matrix = np.zeros((n_bins, n_bins))
        corr_matrix[np.ix_(filtered_bins[chrom], filtered_bins[chrom])] = matrix_C[chrom]

        x_pca = np.zeros(n_bins)
        x_pca[filtered_bins[chrom]] = xpca_dict[chrom]# / np.std(xpca_dict[chrom])
        plot_compartment(corr_matrix,x_pca,n_bins,ax,
                         subplot_spec=cur_gs,title=chrom,vmin=vmin,vmax=vmax,cmap=cmap,ylabel=None)
        col+=1

    if col >= ncols:
        col=0
        row+=1
    cur_gs=gs[row,col]
    ax = ax.figure.add_subplot(cur_gs)
    mm2inch = 1 / 25.4
    legend_width_mm=1 #mm

    ax_extent=ax.get_window_extent()
    legend_width_pixel=legend_width_mm * mm2inch * ax.figure.dpi
    legend_width_ratio=legend_width_pixel/ax_extent.width
    left=((ax_extent.x1+ax_extent.x0) / 2) / ax.figure.get_window_extent().width
    # left=ax_extent.x1 / ax.figure.get_window_extent().width
    bottom=ax_extent.y1 *0.1 / ax.figure.get_window_extent().height
    height=ax_extent.y1 *0.5 / ax.figure.get_window_extent().height
    print(left,bottom,legend_width_ratio,height)

    cax = ax.figure.add_axes(rect=[left,bottom,legend_width_ratio,height],xmargin=0, ymargin=0)  # unit is fractions of figure width and height, left,bottom,width,height
    # cax = inset_axes(ax, loc="center left", borderpad=0,width="10%", height="30%")
    m = plt.cm.ScalarMappable(norm=matplotlib.colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    cbar = ax.figure.colorbar(m, cax=cax,label="Pearson correlation",ticks=[vmin,np.round((vmin+vmax)/2,2), vmax])
    cax.set_ylim([vmin, vmax])
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir,'merged_compartment.pdf'), transparent=True,bbox_inches='tight',dpi=300)
    plt.show()
    print(vmin,vmax)

No. of genes for each chromosome in mm10:
    {'chr7': 2637, 'chr2': 2446, 'chr11': 2095, 'chr5': 1870, 'chr1': 1807, 'chr9': 1802, 'chr4': 1749, 'chr10': 1576, 'chr8': 1571, 'chr6': 1570, 'chr3': 1537, 'chr17': 1471, 'chr13': 1440, 'chr14': 1338, 'chr15': 1147, 'chr12': 1129, 'chrX': 1110, 'chr19': 965, 'chr16': 949, 'chr18': 823, 'chrY': 726, 'chrM': 13}

## Saddle plot of merged raw pc transformed compartment

The difference of the  scores reflected whether the contact was intercompartmental or intracompartmental, and the larger difference represented intercompartmental. The summation of the scores distinguished whether the contact was AA or BB for intracompartmental contacts, and the large positive summation represents the AA interaction, whereas the small negative summation represents the BB interaction. Note that longerrange contained more intercompartmental interactions than shorter-range in general, so non-neuronal cells which had more longerrange interactions also h ad more intercompartmental contacts than neurons when counting the raw contact counts

In [ ]:
if stratify_col is None:
    print(f"Load `filtered_bins` and `pc_loading_dict` from {figdir}")
    filtered_bins = np.load(os.path.join(figdir,'filtered_bins.npz'), allow_pickle=True)
    pc_loading_dict = np.load(os.path.join(figdir,'pc_loadings.npz'), allow_pickle=True)
else: # filtered_bins and pc_loading_dict should have already been generated under stratify_col subfolder
    if stratify_figdir is None:
        stratify_indir=os.path.join(workdir,"compartment",f"{mode}_{stratify_col}")
    else:
        stratify_indir=stratify_figdir
    stratify_indir=os.path.expanduser(stratify_indir)
    print(f"Load `filtered_bins` and `pc_loading_dict` from {indir}")
    filtered_bins = np.load(os.path.join(stratify_indir,'filtered_bins.npz'), allow_pickle=True)
    pc_loading_dict = np.load(os.path.join(stratify_indir,'pc_loadings.npz'), allow_pickle=True)

In [ ]:
def comp_saddle(cool):
    sad = np.zeros((50, 50))
    count = np.zeros((50, 50))
    comptmp = []
    for k,chrom in enumerate(chrom_sizes.index):
        Q = cool.matrix(balance=False, sparse=True).fetch(chrom).toarray() # Q for each chromosome of each cell type
        Q = Q - np.diag(np.diag(Q))
        pc = np.zeros(Q.shape[0])
        binfilter = filtered_bins[chrom] #get filtered bins from merged cell types compartment
        Q = Q[binfilter][:, binfilter]
        decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])])
        E = np.zeros(Q.shape)
        row, col = np.diag_indices(E.shape[0])
        E[row, col] = 1
        for i in range(1, E.shape[0]):
            E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5)
        E = E + E.T #normalized contact map
        C = np.corrcoef(np.log2(E + 0.001)) #correlation matrix
        # transform X using saved PCA model: pc = np.dot(adata.X, pc_loading)
        pc = (C-np.mean(C, axis=0)).dot(pc_loading_dict[chrom]) #use the merged PCA to transform the correlation matrix of each cell type
        comptmp.append(pc)
        labels, groups = pd.qcut(pc, 50, labels=False, retbins=True) ##pd.qcut(x,q=quantile,),return category and bins, divide pc into 50 eaqual size bins
        sad += np.array([[E[np.ix_(labels==i, labels==j)].sum() for i in range(50)] for j in range(50)])
        count += np.array([[(labels==i).sum()*(labels==j).sum() for i in range(50)] for j in range(50)])
    # return np.concatenate(comptmp), sad,count
    return np.concatenate(comptmp), sad/count # for all chromosomes

In [ ]:
# sad = np.zeros((len(group_order),50,50)) #for each cell type, divide the whole E matrix into 50*50 cells, sad is the sum of all interaction of the bins belonging to each 50*50 cell
# count = np.zeros((len(group_order),50,50)) #count is the number of cells in E that are aggregated into each 50*50 cell.
saddle_dict={}
comp = [] # compartment score, transformed X_pca from correlation matrix

In [ ]:
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    if stratify_col is None:
        print(group_order)
        for t,ct in enumerate(group_order):
            sname=ct.replace(' ','_')
            if mode=='impute':
                cool = cooler.Cooler(os.path.join(indir,sname,f"{sname}.Q.cool"))
            elif mode=='raw':
                infile=os.path.join(indir,f"{sname}.mcool::resolutions/100000")
                cool = cooler.Cooler(infile)
            future = executor.submit(
                comp_saddle,
                cool=cool,
            )
            futures[future] = ct
    else:
        print(stratify2group)
        for s in stratify2group:
            s_name=s.replace(' ','_')
            for g in stratify2group[s]:
                g_name=g.replace(' ','_')
                if stratify_figdir is None:
                    prefix=f"{s_name}+{g_name}"
                    ct=f"{s}+{g}"
                else:
                    prefix=g_name
                    ct=g
                if mode=='impute':
                    cool = cooler.Cooler(os.path.join(indir,prefix,f"{prefix}.Q.cool"))
                elif mode=='raw':
                    infile=os.path.join(indir,f"{prefix}.mcool::resolutions/100000")
                    cool = cooler.Cooler(infile)
                future = executor.submit(
                    comp_saddle,
                    cool=cool,
                )
                futures[future] =  ct

    for future in as_completed(futures):
        ct = futures[future] # not dot "." in ct
        pc, sad = future.result()
        comp.append(pd.Series(pc, name=ct)) #transformed X_pca
        saddle_dict[ct]=sad
        print(ct,end=',')

In [ ]:
bins_df = cool.bins()[:] # cool for different cell types have the same bin_df
bins_df.index = bins_df['chrom'].astype(str) + '-' + (bins_df['start'] // res).astype(str)
compidx = np.concatenate([bins_df.query("chrom==@chrom").loc[filtered_bins[chrom]].index for chrom in chrom_sizes.index])

In [ ]:
bins_df.head()

In [ ]:
comp = pd.concat(comp, axis=1) #join the columns, rows are the bins, staying unchanged
comp.index = compidx
comp=comp.loc[:,group_order]
comp.to_csv(os.path.join(figdir,'compartment_score_pcs.tsv.gz'),sep='\t')

In [ ]:
comp

In [ ]:
# sad = sad / count
# np.save(os.path.join(figdir,f'saddle.npy'), sad)
np.savez_compressed(os.path.join(figdir,'saddle.npz'), **saddle_dict)

saddle_strength, which can be thought of as taking the ratio between (AA+BB) / (AB+BA). This corresponds visually to the ratio between the upper left and lower right corners, versus the lower left and upper right corners in the plot above. saddle plot is used to show how “interactive” genomic regions belonging to the A or B compartment are. To calculate the enrichment profile, first all genomic regions are divided into bins, according to their EV value (the “compartment strength”). Then, we use the O/E matrix the average O/E value between all region bins, and take the log2 of the result. Everything is summarised in a matrix, where rows and columns correspond to the genomic region bins, and matrix entries reflect the bins’ interactivity. Positive values reflect more than expected contacts, while negative values reflect less than expected contacts.</br>
Within each chromosome, we ranked all the 100-kb bins based on compartment scores, and grouped the bins into 50 equal-interval groups. The distance-normalized interaction strength between each pair of bins or the PCC of mCG or mCH levels between each pair of bins were averaged within each group. The axes were ranked by the compartment score of the cell types so that BB interactions are on the top left and AA interactions are on the bottom right.

Saddle strength refers to the magnitude or intensity of the dip (saddle) at the center of the Hi-C matrix. A stronger saddle means a more distinct compartmentalization between A and B compartments, with clear regions of the matrix showing high interaction within compartment A and low interaction within compartment B (near the diagonal).

In [ ]:
saddle_dict=np.load(os.path.join(figdir,'saddle.npz'),allow_pickle=True)
saddle_dict

In [ ]:
R=[]
for extent in range(1,26):
    for ct in group_order:
        saddle_strength =[ct,extent,saddle_dict[ct][:extent, :extent].mean(), saddle_dict[ct][-extent:, -extent:].mean(), saddle_dict[ct][:extent, -extent:].mean(), saddle_dict[ct][-extent:, :extent].mean(),
            (saddle_dict[ct][:extent, :extent].sum() + saddle_dict[ct][-extent:, -extent:].sum()) / (saddle_dict[ct][:extent, -extent:].sum() + saddle_dict[ct][-extent:, :extent].sum()),
             saddle_dict[ct][:extent, :extent].sum() / saddle_dict[ct][-extent:, -extent:].sum()]
        R.append(saddle_strength)
saddle_strength = pd.DataFrame(R, columns=['group','extent','BB', 'AA', 'BA', 'AB', 'strength','BB/AA'])
# saddle_strength['mean BB/AA']=saddle_strength.BB / saddle_strength.AA
# BB: upper left corner; AA: lower right; BA: upper right; AB: lower left;
# stregth: intra / inter
if not stratify_col is None:
    if stratify_figdir is None:
        saddle_strength[group_col]=saddle_strength.group.apply(lambda x:x.split('+')[1])
        saddle_strength[stratify_col]=saddle_strength.group.apply(lambda x:x.split('+')[0])
    else:
        group2stratify=metadata.loc[:,[group_col,stratify_col]].drop_duplicates().set_index(group_col)[stratify_col].to_dict()
        saddle_strength[group_col]=saddle_strength.group
        saddle_strength[stratify_col]=saddle_strength[group_col].map(group2stratify)
    saddle_strength[stratify_col]=saddle_strength[stratify_col].apply(lambda x:x.replace('_',' '))
    saddle_strength.to_csv(os.path.join(figdir,'compartment_score_saddle_strength.tsv.gz'),sep='\t')
compstr=saddle_strength.loc[saddle_strength.extent==10]
compstr.set_index('group',inplace=True)
compstr.sort_values('strength',inplace=True)
saddle_strength.to_csv(os.path.join(figdir,'compartment_score_saddle_strength.tsv'),sep='\t')
saddle_strength

In [ ]:
if stratify_col is None:
    palette=group_colors.copy()
    palette['merged']='black'
    hue='group'
else:
    palette=stratify_colors
    hue=stratify_col
plt.figure(figsize=(3.5,3))
ax=sns.lineplot(data=saddle_strength,x='extent',y='strength',hue=hue,palette=palette)
handles, labels = ax.get_legend_handles_labels()
l = plt.legend(handles, labels,bbox_to_anchor=(1, 0),
               loc='lower left',ncol=1,labelspacing=0.2,handleheight=0.5)
plt.savefig(os.path.join(figdir,f'saddle_stregnth_lineplot.pdf'), transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
df=saddle_strength.pivot(index='group',columns='extent',values='strength')
plt.figure(figsize=(5,4))
if not parent_col is None:
    df_rows=df.index.to_series().map(group2parent).to_frame(name=parent_col)
    row_ha_dict={
                parent_col:anno_simple(df_rows[parent_col],
                                         colors=parent_colors,legend=True),
                }
    row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
             verbose=0,label_kws=dict(visible=False))
    row_split=df_rows[parent_col]
else:
    row_ha=None
    row_split=None
ClusterMapPlotter(df,col_cluster=False,row_cluster=False,#annot=True,fmt=".1g",
                  left_annotation=row_ha,row_split=row_split,
                 show_rownames=True,show_colnames=True,
                  yticklabels=True,label='strength',
                 verbose=0)
plt.savefig(os.path.join(figdir,f'saddle_stregnth_heatmap.pdf'), transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
df=saddle_strength.pivot(index='group',columns='extent',values='BB/AA')
plt.figure(figsize=(5,4))
if not parent_col is None:
    df_rows=df.index.to_series().map(group2parent).to_frame(name=parent_col)
    row_ha_dict={
                parent_col:anno_simple(df_rows[parent_col],
                                         colors=parent_colors,legend=True),
                }
    row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
             verbose=0,label_kws=dict(visible=False))
    row_split=df_rows[parent_col]
else:
    row_ha=None
    row_split=None
ClusterMapPlotter(df,col_cluster=False,row_cluster=False,#annot=True,fmt=".1g",
                  left_annotation=row_ha,row_split=row_split,
                 show_rownames=True,show_colnames=True,
                  yticklabels=True,label='BB / AA',vmax=1.5,vmin=0.5,
                 cmap='coolwarm',verbose=0)
plt.savefig(os.path.join(figdir,f'saddle_BB_AA_ratio_heatmap.pdf'), transparent=True,bbox_inches='tight')
plt.show()

The BB/AA ratio is a way of quantifying the degree of compartmentalization between active (A) and inactive (B) chromatin.
A higher BB/AA ratio indicates a more closed and less compartmentalized genome, where there are relatively more interactions within inactive chromatin regions (compartment B) compared to interactions within active regions (compartment A).
A lower BB/AA ratio suggests that the genome is more organized with a clear distinction between active and inactive chromatin, and more interactions occur within compartment A.

In [ ]:
compstr.sort_values('BB/AA',ascending=True)

In [ ]:
# barplot for BB/AA ratio
if stratify_col is None:
    x='group'
else:
    x=group_col
palette=group_colors.copy()
palette['merged']='black'
plt.figure(figsize=(5,3))
ax=sns.barplot(data=compstr.sort_values('BB/AA',ascending=True),x=x,y='BB/AA',hue=None,palette=palette)
ax.set_ylim(0.8,1.1)
ax.xaxis.label.set_visible(False)
ax.set_ylabel("BB/AA ratio",fontsize=14)
plt.setp(plt.gca().get_xticklabels(), rotation=-45, ha='left',rotation_mode='anchor')
plt.savefig(os.path.join(figdir,f'saddle_BB_AA_ratio_barplot.pdf'), transparent=True,bbox_inches='tight',dpi=300)
plt.show()
# compstr.sort_values('BB/AA',ascending=True).plot(y='BB/AA',kind='bar',)

In [ ]:
def plot_saddle(cell_types,saddle_dict,strength_dict,compstr,stratify_col,
            parent_colors,group2parent,parent_col,group_colors,
            figdir,stratify):
    # from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    vmin, vmax = -0.8, 0.8
    mm2inch = 1 / 25.4
    legend_width_mm=1 #mm
    cmap='RdYlBu_r' #coolwarm
    N=len(cell_types)+1
    ncols=min(N,6)
    nrows=int(np.ceil(N / ncols))
    print(nrows,ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3,nrows*3), dpi=300,
                             gridspec_kw=dict(wspace=0.15,hspace=0.2))
    for i,ct in enumerate(cell_types):
        if isinstance(axes,np.ndarray):
            ax = axes.flatten()[i]
        else:
            ax=axes
        plot = ax.imshow(np.log2(saddle_dict[ct]+0.001), cmap=cmap, vmin=vmin, vmax=vmax,rasterized=True)
        ax.set_xticks([])
        if stratify_col is None:
            color=parent_colors[group2parent[ct]] if not parent_col is None else 'black'
            title=ct
        else:
            g2=ct.split('+')[1]
            # color=stratify_colors[group2stratify[ct]]
            color=group_colors[g2]
            title=g2
        # ax.set_title(f"{title}\n({strength_dict[ct]})", fontsize=15,color=color,fontweight='bold')
        ax.set_title(title, fontsize=14,color=color,fontweight='bold') #strength_dict[ct]
        ax.text(0.5, 0.5, np.around(strength_dict[ct], decimals=3),color='blue',
                ha='center', va='center', fontsize=15,transform=ax.transAxes) #lower right / lower left
        ax.text(0.95, 0.05, np.around(compstr.loc[ct, 'AA']/compstr.loc[ct, 'AB'], decimals=3),
                ha='right', va='bottom', fontsize=15,transform=ax.transAxes) #lower right / lower left
        ax.text(0.05, 0.95, np.around(compstr.loc[ct, 'BB']/compstr.loc[ct, 'BA'], decimals=3),
                ha='left', va='top', fontsize=15,transform=ax.transAxes) #upper left / upper right

    ax=axes.flatten()[i+1]
    ax_extent=ax.get_window_extent()
    legend_width_pixel=legend_width_mm * mm2inch * ax.figure.dpi
    legend_width_ratio=legend_width_pixel/ax_extent.width
    # left=((ax_extent.x1+ax_extent.x0) / 2) / ax.figure.get_window_extent().width
    left=ax_extent.x0 / ax.figure.get_window_extent().width
    height=ax_extent.height *0.8 / ax.figure.get_window_extent().height # from top to bottom, left to right
    bottom=(ax_extent.y0 + ax_extent.height *0.1) / ax.figure.get_window_extent().height # from top to bottom
    print(ax_extent.get_points(),ax.figure.get_window_extent().get_points()) # get_points return (x0,y0) (x1,y1)
    print(left,bottom,legend_width_ratio,height,ax_extent.y0,ax_extent.y1)
    cax = ax.figure.add_axes(rect=[left,bottom,legend_width_ratio,height],xmargin=0, ymargin=0)  # unit is fractions of figure width and height, left,bottom,width,height
    # cax = inset_axes(ax, loc="lower left", borderpad=0, width="10%", height="80%")
    m = plt.cm.ScalarMappable(norm=matplotlib.colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    cax.set_ylim([vmin, vmax])
    cbar = ax.figure.colorbar(m, cax=cax,ax=ax,ticks=[-0.8,-0.4,0,0.4,0.8], label="Normalized Interaction") #'rotation': 270, 'labelpad': 15
    cbar.ax.yaxis.label.set_fontsize(14)
    ax.axis('off')
    if isinstance(axes,np.ndarray):
        for ax in axes.flatten()[len(cell_types):]:
                ax.axis('off')
    # cbar = plt.colorbar(plot, ax=ax)
    # cbar.set_ticks([vmin,vmax])
    # cbar.set_label('Normalized Interaction')
    # plt.subplots_adjust(left=0,right=1,bottom=0,top=1,wspace=0,hspace=0)
    # plt.tight_layout()
    if stratify_col is None:
        outname=os.path.join(figdir,'celltype_saddle.pdf')
    else:
        outname=os.path.join(figdir,f'{stratify}_saddle.pdf')
    plt.savefig(outname, transparent=True,bbox_inches='tight')

In [ ]:
strength_dict=compstr.strength.apply(lambda x:round(x,2)).to_dict()
if stratify_col is None:
    plot_saddle(group_order,saddle_dict,strength_dict,compstr,stratify_col,
                parent_colors,group2parent,parent_col,group_colors,
                figdir,stratify=None)
else:
    for stratify in stratify2group:
        if stratify_figdir is None:
            cell_types=[f"{stratify}+{ct}" for ct in stratify2group[stratify] if f"{stratify}+{ct}" in saddle_dict]
        else:
            cell_types=[ct for ct in stratify2group[stratify] if ct in saddle_dict]
        if len(cell_types)<=1:
            continue
        plot_saddle(cell_types,saddle_dict,strength_dict,compstr,stratify_col,
            parent_colors,group2parent,parent_col,group_colors,
            figdir,stratify=stratify)

The upper-left and bottom-right of the saddle plot represent the contact frequency between B-B and A-A compartments, the upper-right and bottom left of the saddle plot represent the contact frequency between A and B compartments. </br>
saddle_strength, which can be thought of as taking the ratio between (AA+BB) / (AB+BA), intra / inter

## Plot the Q matrix after filtering and compartments for major cell types

In [ ]:
if stratify_col is None:
    print(f"Load `filtered_bins` and `comp` from {figdir}")
    # filtered_bins = np.load(os.path.join(figdir,'filtered_bins.npz'), allow_pickle=True)
    filtered_bins_path=os.path.join(figdir,'filtered_bins.npz')
else: # filtered_bins and pc_loading_dict should have already been generated under stratify_col subfolder
    if stratify_figdir is None:
        stratify_indir=os.path.join(workdir,"compartment",f"{mode}_{stratify_col}")
    else:
        stratify_indir=stratify_figdir
    stratify_indir=os.path.expanduser(stratify_indir)
    print(f"Load `filtered_bins` and `comp` from {stratify_indir}")
    # filtered_bins = np.load(os.path.join(stratify_indir,'filtered_bins.npz'), allow_pickle=True)
    filtered_bins_path=os.path.join(stratify_indir,'filtered_bins.npz')

In [ ]:
# for each chrom
def load_filtered_Q(celltypes,chrom,indir,filtered_bins_path,mode):
    filtered_bins = np.load(filtered_bins_path, allow_pickle=True)
    binfilter = filtered_bins[chrom] #filtered bins for current chromosome
    filtered_Q = {} #store the filtered Q (remove some bins with abnormal coverage)
    for ct in celltypes:
        sname=ct.replace(' ','_')
        if mode=='impute':
            infile=os.path.join(indir,sname,f"{sname}.Q.cool") #read imputed Q for each major type, indir is impute 100k compartment dir (imputed_100K_major_type_dir)
        else: # raw
            infile=os.path.join(indir,f"{sname}.mcool::resolutions/100000")
        cool = cooler.Cooler(infile) # read raw Q for each major cell type
        Q = cool.matrix(balance=False, sparse=True).fetch(chrom).toarray()
        Q = Q - np.diag(np.diag(Q)) #interactionn
        # rowsum = Q.sum(axis=0)
        Q = Q[binfilter][:, binfilter]
        filtered_Q[ct]=Q
    return filtered_Q,binfilter

def plot_chrom_Q(celltypes,binfilter,n_bins,filtered_Q,figdir1,outname):
    nrows=int(np.ceil(len(celltypes)/6))
    fig, axes = plt.subplots(nrows, 6, figsize=(18,nrows*3), dpi=300,gridspec_kw=dict(wspace=0.25,hspace=0.2))
    for i,ct in enumerate(celltypes):
        ax = axes.flatten()[i]
        Q1 = np.zeros((n_bins, n_bins))
        Q1[np.ix_(binfilter, binfilter)] = filtered_Q[ct]
        vmax=np.percentile(filtered_Q[ct], 95)
        ax.imshow(Q1, cmap='bwr',vmin=-1*vmax, vmax=vmax,rasterized=True)
        ax.set_xticks([])
        vmax=round(vmax,2)
        ax.set_title(f"{ct} ({vmax})", fontsize=14)
        # print(ct,vmax,end=';')
    for ax in axes.flatten()[len(celltypes):]:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir1,outname), transparent=True,bbox_inches='tight',dpi=300)
    plt.close(ax.figure)

def plot_chrom_compartment(chrom,celltypes,matrix_C,xpca_dict,binfilter,n_bins,
                           stratify_col,parent_colors,group2parent,parent_col,
                          stratify_colors,figdir1,outname):
    # Plot compartment for major cell types in one chrom
    N=len(celltypes)+1
    ncols=min(N,6)
    nrows=int(np.ceil(N / ncols))
    matrix_all=[]
    for ct in celltypes:
        matrix_all.append(matrix_C[ct].flatten())
    all_contacts=np.hstack(matrix_all)
    del matrix_all
    v=max(abs(np.percentile(all_contacts,5)),abs(np.percentile(all_contacts,95)))
    vmin,vmax=-1*v,v #use the same vmin and vmax for all chromosomes
    del all_contacts
    cmap='BlueDarkRed12'

    plt.clf()
    plt.figure(figsize=(3*ncols+0.2,3.6*nrows),dpi=300)
    ax=plt.gca()
    ax.set_axis_off()
    gs = ax.figure.add_gridspec(nrows,ncols,wspace=0.15,hspace=0.16) # width_ratios=self.widths,height_ratios=self.heights,

    row,col=0,0
    for k,ct in enumerate(celltypes):
        if col >= ncols:
            col=0
            row+=1
        # if col==0:
        #     ylabel=chrom
        # else:
        #     ylabel=None
        cur_gs=gs[row,col]
        corr_matrix = np.zeros((n_bins, n_bins))
        corr_matrix[np.ix_(binfilter, binfilter)] = matrix_C[ct]
        x_pca = np.zeros(n_bins)
        x_pca[binfilter] = xpca_dict[ct]# / np.std(xpca_dict[i])
        if stratify_col is None:
            color=parent_colors[group2parent[ct]] if not parent_col is None else 'black'
        else:
            color=stratify_colors[ct.split('+')[0].replace('_',' ')]
        plot_compartment(corr_matrix,x_pca,n_bins,ax,
                         subplot_spec=cur_gs,title=ct,vmin=vmin,vmax=vmax,cmap=cmap,color=color,ylabel=None)
        col+=1

    if col >= ncols:
        col=0
        row+=1
    cur_gs=gs[row,col]
    # gs = matplotlib.gridspec.GridSpecFromSubplotSpec(2,1,subplot_spec=subplot_spec,height_ratios=[3,0.4],hspace=0.0,wspace=0)
    ax = ax.figure.add_subplot(cur_gs)
    mm2inch = 1 / 25.4
    legend_width_mm=1 #mm

    ax_extent=ax.get_window_extent()
    legend_width_pixel=legend_width_mm * mm2inch * ax.figure.dpi
    legend_width_ratio=legend_width_pixel/ax_extent.width
    # left=((ax_extent.x1+ax_extent.x0) / 2) / ax.figure.get_window_extent().width
    left=ax_extent.x0 / ax.figure.get_window_extent().width
    bottom=ax_extent.y1 *0.1 / ax.figure.get_window_extent().height
    height=ax_extent.y1 *0.5 / ax.figure.get_window_extent().height
    # print(left,bottom,legend_width_ratio,height)
    cax = ax.figure.add_axes(rect=[left,bottom,legend_width_ratio,height],xmargin=0, ymargin=0)  # unit is fractions of figure width and height, left,bottom,width,height
    m = plt.cm.ScalarMappable(norm=matplotlib.colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    cbar = ax.figure.colorbar(m, cax=cax,label="Pearson correlation",ticks=[vmin,np.round((vmin+vmax)/2,2), vmax])
    cax.set_ylim([vmin, vmax])
    cbar.ax.yaxis.label.set_fontsize(14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir1,outname), transparent=True,bbox_inches='tight',dpi=300)
    plt.close(ax.figure)
    # print(vmin,vmax)

def plot_chrom(chrom,chrom_sizes,res,filtered_bins_path,comp_score_path,stratify2group,group_order,stratify_col,
               figdir,indir,mode):
    # print(chrom)
    comp = pd.read_csv(comp_score_path, sep='\t',index_col=0)
    n_bins = (int(chrom_sizes.loc[chrom]) // res + 1)
    if stratify_col is None:
        celltypes=group_order
    else:
        celltypes=[]
        for s in stratify2group:
            for g in stratify2group[s]:
                ct=f"{s}+{g}"
                celltypes.append(ct)
    filtered_Q,binfilter=load_filtered_Q(celltypes,chrom,indir,filtered_bins_path,mode)
    if stratify_col is None:
        plot_chrom_Q(group_order,binfilter,n_bins,filtered_Q,figdir,f'{chrom}_Q.pdf')
    else:
        figdir1=os.path.join(figdir,chrom)
        if not os.path.exists(figdir1):
            os.makedirs(figdir1,exist_ok=True)
        for s in stratify2group:
            celltypes=[f"{s}+{g2}" for g2 in stratify2group[s]]
            outname=f'{s}_{chrom}_Q.pdf'
            plot_chrom_Q(celltypes,binfilter,n_bins,filtered_Q,figdir1,outname)

    ## Calculate eigenvector for major types
    matrix_C = {} #correlation coefficient matrixes for each major cell type
    xpca_dict = {} # a list of transformed PC for all major cell types
    for k,ct in enumerate(group_order):
        Q = filtered_Q[ct].copy()
        decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])])
        E = np.zeros(Q.shape)
        row, col = np.diag_indices(E.shape[0])
        E[row, col] = 1
        for i in range(1, E.shape[0]):
            E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5)
        E = E + E.T #normalized Q: normalized interaction
        C = np.corrcoef(np.log2(E + 0.001)) #Pearson product-moment correlation coefficients. matrix
        matrix_C[ct]=C
        xpca = comp.loc[(comp.index.str.split('-').str[0]==chrom), ct] #transformed X_PCA from the correlation matrix using the merged cells (all major cell types)
        xpca.index = [int(xx.split('-')[1]) for xx in xpca.index] #bins
        xpca_dict[ct]=xpca[np.where(binfilter)[0]].values #keep only the filtered bins
        # print(ct,end=',')
    if stratify_col is None:
        plot_chrom_compartment(chrom,group_order,matrix_C,xpca_dict,binfilter,n_bins,
                           stratify_col,parent_colors,group2parent,parent_col,
                          dict(),figdir,f'{chrom}_compartment.pdf')
    else:
        figdir1=os.path.join(figdir,chrom)
        if not os.path.exists(figdir1):
            os.makedirs(figdir1,exist_ok=True)
        for s in stratify2group:
            # print(s)
            celltypes=[f"{s}+{g2}" for g2 in stratify2group[s]]
            outname=f'{s}_{chrom}_compartment.pdf'
            plot_chrom_compartment(chrom,celltypes,matrix_C,xpca_dict,binfilter,n_bins,
                           stratify_col,parent_colors,group2parent,parent_col,
                          stratify_colors,figdir1,outname)
    return chrom

In [ ]:
comp_score_path=os.path.join(figdir,'compartment_score_pcs.tsv.gz')
# comp = pd.read_csv(comp_score_path, sep='\t',index_col=0)
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for chrom in chrom_sizes.index:
        future = executor.submit(
            plot_chrom,
            chrom=chrom,chrom_sizes=chrom_sizes,res=res,filtered_bins_path=filtered_bins_path,
            comp_score_path=comp_score_path,stratify2group=stratify2group,group_order=group_order,
            stratify_col=stratify_col,figdir=figdir,indir=indir,mode=mode
        )
        futures[future] = chrom

    for future in as_completed(futures):
        chrom = futures[future]
        r = future.result()
        print(r,end=',')